# Week 1.4: Prompt Engineering for Digital Twins

## Learning Objectives
- Master different prompting techniques
- Understand zero-shot, few-shot, and chain-of-thought
- Apply prompting to digital twin interactions

## Key Insight
> 'Moving from Chat to System Instructions'

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict

print('✅ Ready for prompting exercises!')

✅ Ready for prompting exercises!


## Part 1: Prompting Fundamentals

### 1.1 Zero-Shot Prompting

In [2]:
# Example prompts for digital twin
zero_shot_examples = [
    {
        'task': 'activity_prediction',
        'prompt': 'Predict the next activity: I just finished work, had dinner, and now...',
        'expected': 'leisure/relaxation activity'
    },
    {
        'task': 'preference_extraction',
        'prompt': 'Extract preferences from: I love outdoor activities but hate crowded places',
        'expected': {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}
    }
]

print('Zero-Shot Examples:')
for ex in zero_shot_examples:
    print(f"\nTask: {ex['task']}")
    print(f"Prompt: {ex['prompt']}")
    print(f"Expected: {ex['expected']}")

Zero-Shot Examples:

Task: activity_prediction
Prompt: Predict the next activity: I just finished work, had dinner, and now...
Expected: leisure/relaxation activity

Task: preference_extraction
Prompt: Extract preferences from: I love outdoor activities but hate crowded places
Expected: {'likes': ['outdoor', 'quiet'], 'dislikes': ['crowds']}


### 🎯 Exercise 1.1 (Easy): Create Your Prompts

**Task**: Design prompts for your digital twin use case.

In [ ]:
# YOUR CODE HERE
my_prompts = [
     {
        'task': 'activity_prediction',
        'prompt': 'Predict the next activity: I just finished work, had dinner, and now...',
        'expected': 'leisure/relaxation activity'
    },
    {
        'task': 'preference_extraction',
        'prompt': 'Extract preferences from: I love outdoor activities but hate crowded places',
        'expected': {'likes': ['garden', 'outdoor', 'quiet'], 'dislikes': ['crowds', 'shops']}
    }
]

In [4]:
!pip install -q groq

In [26]:
import os, json
from groq import Groq
from google.colab import userdata

api_key=userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)
model="openai/gpt-oss-120b"

def llm(prompt: str) -> str:
    """Simple helper for single-turn prompts"""
    chat_completion = client.chat.completions.create(
        #model="llama-3.3-70b-versatile",
        #model="openai/gpt-oss-120b",
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        #temperature=0.7,
        #max_tokens=1024,
        #stream=True,
    )
    # Extract and return the text content from the response message
    return chat_completion.choices[0].message.content

print("Connected to Groq. Ready.")


Connected to Groq. Ready.


#Zero-Shot. few-shot, and chain of thought prompting

In [16]:
#Zero-shot: describe the task only
#No examples, no format instructions
#Model infers the answer from training

prompt = """
Predict the next activity.
I just finished work, had dinner,
and now
"""

response = llm(prompt)
print(response)

Sounds like you’re winding down after a long day. A common next step after dinner is to settle into something relaxing—maybe you’ll:

- **Kick back in front of the TV** and catch up on a favorite show or a movie.  
- **Scroll through your phone** (social media, news, or a quick game) while you unwind.  
- **Grab a good book** or a tablet and read a few chapters.  
- **Take a short walk** around the block to digest and get a bit of fresh air.  
- **Start a hobby**—perhaps knitting, drawing, or playing a musical instrument.  
- **Prepare for tomorrow** by checking your calendar or laying out clothes.  
- **Head straight to bed** if you’re feeling especially tired.

If any of those hit the mark, let me know! If you’ve got something else in mind, I’m curious to hear what you’re about to do.


In [18]:
prompt = """
Predict the next activity.
I just went outside to the garden,
and now
"""

response = llm(prompt)
print(response)

You’re probably about to start taking care of the plants—maybe grabbing a watering can and giving the flower beds a good soak, or checking on the veggies you’ve been growing. 🌱💧


In [22]:
#Few-shot: show examples, do not explain
#2-3 input/output pairs before the real question
#Model leanrs format from demonstration

prompt = """
Examples:
Input: Finished gym, lunch, coffee
Output: {"next:"Work","confidence":0.82}

Input: Dinner, watched TV
Output: {"next":"Sleep","confidence":0.91}

Input: Finished work, had dinner, now
Output:
"""

response = llm (prompt)
print(response)

{"next":"Relax","confidence":0.78}


In [23]:
#Chain-of-thought: four extra words
#Append: Think step by step.
#Model shifts to explicit reasoning

plain = """
Should I exercsie today?
Context: worked 10h, slept 5h.
"""

cot = """
Should I exercise today?
Context: worked 10h, slept 5h.
Think step by step.
"""

print("--- PLAIN ---")
print(llm(plain))
print()
print("--- CHAIN-OF-THOUGHT ---")
print(llm(cot))

--- PLAIN ---
### Quick “Should I work out?” check‑in

| ✅ | ✅ | ✅ |
|---|---|---|
| **You worked 10 h** | **You only got ~5 h of sleep** | **You’re wondering about exercise** |

---

## 1️⃣ What your body is probably saying right now
- **Low sleep** → reduced reaction time, lower glycogen stores, higher perceived effort, and a modest dip in immune function.  
- **Long work shift** → accumulated mental and (often) physical stress, plus a build‑up of cortisol.

If you push into a **high‑intensity** session (heavy weights, HIIT, long run) you may:
- Feel unusually “flat” or shaky.
- Increase risk of minor injuries (muscle strains, joint irritation) because coordination and proprioception are blunted when you’re sleep‑deprived.
- Compromise recovery, making the next day feel even worse.

That doesn’t mean you have to skip movement altogether—just **tune the intensity and duration** to match how you feel.

---

## 2️⃣ A “smart‑size” workout for a 5‑hour night

| Goal | Example (≈15–30 min)

# Build the Twin's System Instruction

In [27]:
#Build the Twin's System Instruction

system = """
ROLE: You are a digital twin for {user}.
You represent their preferences and
behavioural patterns.

KNOWLEDGE: You have the user's activity
history and temporal patterns.

CONSTRAINTS: Always return structured
output. Never guess. If data is missing,
say so explicitly.

TONE: Precise, analytical.
"""

r = client.chat.completions.create(
    model=model,
    messages=[
        {'role':'system','content':system},
        {'role':'user','content':'What next?'}
    ]
)
print(r.choices[0].message.content)


{
  "response": null,
  "reason": "Insufficient contextual data to determine the appropriate next step.",
  "required_data": [
    "User's recent activity history",
    "Current task or project context",
    "Temporal patterns or schedule information"
  ],
  "suggestion": "Provide details about your recent actions or the specific domain you are referring to so that a precise recommendation can be generated."
}


In [32]:
#Build the Twin's System Instruction

user = "Lynton"

user_profile = {
    "name": "Lynton",
    "role": "University Lecturer",
    "subject expert": "Pathophysiology",
    "preferences": [
        "prefers analytical explanations",
        "like structres responses",
        "works often with Python and presentations"
    ],
    "goals": [
        "improvde teaching materials",
        "prepare creative lectures",
        "build research workflows",
    ]
}

activity_history = [
    {"time": "2026-07-10", "activity": "Created slides on cardiovascualr disease"},
    {"time": "2026-07-10", "activity": "Worked on Python code for project"},
    {"time": "2026-07-10", "activity": "Prepared AI enabled assessments of cardiovascular disease"},
]

temporal_patterns = {
    "most_active_period": "mornings",
    "common_tasks": [
        "lecture preparation",
        "report writing",
        "coding examples"
    ],
    "recurring_focus": [
        "artificial intelligence",
        "electrocardiography"
    ]
}

system = f"""
ROLE: You are a digital twin for {user}.
You simulate the user's likely priorities, preferences, and next actions
using only the evidence provided.

AVAILABLE DATA:
1. User profile:
{user_profile}

2. Activity history:
{activity_history}

3. Temporal patterns:
{temporal_patterns}

OBJECTIVE:
Given a user query, produce an informed and structured assessment of:
- the user's likely current context,
- the most probable next actions,
- the evidence supporting each action,
- any missing data that limits confidence.

REASONING RULES:
- Use only the supplied data.
- Do not invent facts.
- Seperate observed facts from inferred conclusions.
- If evidence is weak or missing, say so explicitly.
- Rank likely next actions by confidence.
- Ground every inference in profile, history, or temporal patterns.

OUTPUT FORMAT:
Return valid JSON with this structure:

{{
    "user": "string",
    "current_context": {{
        "summary": "string",
        "observed_signals": ["string"]
    }},
    "predicted_next_actions": [
        {{
            "action": "string",
            "confidence": "high | medium | low",
            "why": ["string"],
            "supporting_data": ["string"]
        }}
    ],
    "missing_information": ["string"],
    "response_quality": {{
      "grounded_in_data": true,
      "contains_guessing": false
    }}
}}


TONE:
Precise, analytical, concise.
"""

In [33]:
r = client.chat.completions.create(
    model=model,
    messages=[
        {'role':'system','content':system},
        {'role':'user','content':'What next?'}
    ],
    response_format={"type": "json_object"}
)
print(r.choices[0].message.content)

{
  "user": "Lynton",
  "current_context": {
    "summary": "Morning work session focused on cardiovascular disease teaching materials, Python coding, and AI-enabled assessments.",
    "observed_signals": [
      "Created slides on cardiovascular disease (2026-07-10)",
      "Wrote Python code for a project (2026-07-10)",
      "Prepared AI-enabled assessments of cardiovascular disease (2026-07-10)",
      "Profile indicates preference for analytical, structured responses and frequent Python use"
    ]
  },
  "predicted_next_actions": [
    {
      "action": "Integrate Python code examples into the cardiovascular disease slides as interactive Jupyter notebooks",
      "confidence": "high",
      "why": [
        "Recent coding activity aligns with slide creation",
        "Profile shows frequent use of Python for teaching",
        "Morning period is typical for lecture‑preparation tasks"
      ],
      "supporting_data": [
        "Activity history: 'Worked on Python code for project'